In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/protein_grid.parquet')

# Sorting is mandatory before any time-based operation
df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Memory:", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

Shape: (7732111, 11)
Memory: 184.3 MB
Date range: 2015-01-01 to 2017-08-15


In [2]:
d = df["date"].dt

df["dayofweek"]   = d.dayofweek.astype("int8")      # 0=Mon ... 6=Sun
df["day"]         = d.day.astype("int8")
df["month"]       = d.month.astype("int8")
df["year"]        = d.year.astype("int16")
df["weekofyear"]  = d.isocalendar().week.astype("int8")
df["is_weekend"]  = (d.dayofweek >= 5)

# Ecuadorian retail: wages are paid on the 15th and last day of month
df["is_payday"]   = ((d.day == 15) | (d.is_month_end))

# Position within the month, sometimes captures pay-cycle spending curves
df["days_in_month"] = d.days_in_month.astype("int8")

print(df[["date", "dayofweek", "month", "is_weekend", "is_payday"]].head(10))
print("\nNew shape:", df.shape)

        date  dayofweek  month  is_weekend  is_payday
0 2015-01-03          5      1        True      False
1 2015-01-04          6      1        True      False
2 2015-01-05          0      1       False      False
3 2015-01-06          1      1       False      False
4 2015-01-07          2      1       False      False
5 2015-01-08          3      1       False      False
6 2015-01-09          4      1       False      False
7 2015-01-10          5      1        True      False
8 2015-01-11          6      1        True      False
9 2015-01-12          0      1       False      False

New shape: (7732111, 19)


In [3]:
# Build a lag

# Group once, reuse — grouping 7.7M rows repeatedly is expensive
grp = df.groupby(["store_nbr", "item_nbr"], observed=True)["unit_sales"]

# Lags chosen deliberately, not arbitrarily
LAGS = [1, 7, 14, 28]

for lag in LAGS:
    df[f"lag_{lag}"] = grp.shift(lag).astype("float32")

print(df[["store_nbr", "item_nbr", "date", "unit_sales", "lag_1", "lag_7"]].head(10))
print("\nNulls in lag_1:", df["lag_1"].isna().sum())
print("Nulls in lag_28:", df["lag_28"].isna().sum())

   store_nbr  item_nbr       date  unit_sales  lag_1  lag_7
0          1    108696 2015-01-03         1.0    NaN    NaN
1          1    108696 2015-01-04         1.0    1.0    NaN
2          1    108696 2015-01-05         3.0    1.0    NaN
3          1    108696 2015-01-06         2.0    3.0    NaN
4          1    108696 2015-01-07         1.0    2.0    NaN
5          1    108696 2015-01-08         1.0    1.0    NaN
6          1    108696 2015-01-09         0.0    1.0    NaN
7          1    108696 2015-01-10         1.0    0.0    1.0
8          1    108696 2015-01-11         0.0    1.0    1.0
9          1    108696 2015-01-12         1.0    0.0    3.0

Nulls in lag_1: 10009
Nulls in lag_28: 272500


In [4]:
# Creating Rolling windows features

# Rolling stats must be computed on the SHIFTED series to avoid leakage
shifted = grp.shift(1)

WINDOWS = [7, 14, 28]

for w in WINDOWS:
    roll = shifted.groupby([df["store_nbr"], df["item_nbr"]], observed=True).rolling(w, min_periods=1)
    df[f"roll_mean_{w}"] = roll.mean().reset_index(level=[0,1], drop=True).astype("float32")
    df[f"roll_std_{w}"]  = roll.std().reset_index(level=[0,1], drop=True).astype("float32")
    df[f"roll_max_{w}"]  = roll.max().reset_index(level=[0,1], drop=True).astype("float32")

print(df[["date", "unit_sales", "lag_1", "roll_mean_7", "roll_std_7"]].head(10))
print("\nShape:", df.shape)
print("Memory:", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")

        date  unit_sales  lag_1  roll_mean_7  roll_std_7
0 2015-01-03         1.0    NaN          NaN         NaN
1 2015-01-04         1.0    1.0     1.000000         NaN
2 2015-01-05         3.0    1.0     1.000000    0.000000
3 2015-01-06         2.0    3.0     1.666667    1.154701
4 2015-01-07         1.0    2.0     1.750000    0.957427
5 2015-01-08         1.0    1.0     1.600000    0.894427
6 2015-01-09         0.0    1.0     1.500000    0.836660
7 2015-01-10         1.0    0.0     1.285714    0.951190
8 2015-01-11         0.0    1.0     1.285714    0.951190
9 2015-01-12         1.0    0.0     1.142857    1.069045

Shape: (7732111, 32)
Memory: 634.2 MB


In [5]:
# --- Promotion features ---
promo_grp = df.groupby(["store_nbr", "item_nbr"], observed=True)["onpromotion"]

# Was it on promo recently? (promos often run in multi-day blocks)
df["promo_lag_1"] = promo_grp.shift(1).astype("float32")

# How many promo days in the trailing 4 weeks?
promo_shifted = promo_grp.shift(1).astype("float32")
df["promo_count_28"] = (
    promo_shifted.groupby([df["store_nbr"], df["item_nbr"]], observed=True)
    .rolling(28, min_periods=1).sum()
    .reset_index(level=[0,1], drop=True).astype("float32")
)

# --- Intermittency: how sporadic is this item? ---
# Days since this pair last had a non-zero sale
sold = (df["unit_sales"] > 0)
df["days_since_sale"] = (
    df.assign(_s=sold)
      .groupby(["store_nbr", "item_nbr"], observed=True)["_s"]
      .transform(lambda s: s.shift(1).groupby((s.shift(1) == True).cumsum()).cumcount())
      .astype("float32")
)

# Share of trailing 28 days with zero sales
zero_flag = (df["unit_sales"] == 0).astype("float32")
df["zero_rate_28"] = (
    zero_flag.groupby([df["store_nbr"], df["item_nbr"]], observed=True)
    .shift(1).groupby([df["store_nbr"], df["item_nbr"]], observed=True)
    .rolling(28, min_periods=1).mean()
    .reset_index(level=[0,1], drop=True).astype("float32")
)

print(df[["date", "unit_sales", "onpromotion", "promo_lag_1", "promo_count_28", "days_since_sale", "zero_rate_28"]].head(12))
print("\nShape:", df.shape)
print("Memory:", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")

         date  unit_sales  onpromotion  promo_lag_1  promo_count_28  \
0  2015-01-03         1.0        False          NaN             NaN   
1  2015-01-04         1.0        False          0.0             0.0   
2  2015-01-05         3.0        False          0.0             0.0   
3  2015-01-06         2.0        False          0.0             0.0   
4  2015-01-07         1.0        False          0.0             0.0   
5  2015-01-08         1.0        False          0.0             0.0   
6  2015-01-09         0.0        False          0.0             0.0   
7  2015-01-10         1.0        False          0.0             0.0   
8  2015-01-11         0.0        False          0.0             0.0   
9  2015-01-12         1.0        False          0.0             0.0   
10 2015-01-13         1.0        False          0.0             0.0   
11 2015-01-14         1.0        False          0.0             0.0   

    days_since_sale  zero_rate_28  
0               0.0           NaN  
1   

In [6]:
# Handle the nulls and set the training cutoff:

In [7]:
# How bad is it?
print("Null counts by column:")
nulls = df.isna().sum()
print(nulls[nulls > 0])
print(f"\nRows with ANY null: {df.isna().any(axis=1).sum():,} ({df.isna().any(axis=1).mean()*100:.1f}%)")

Null counts by column:
lag_1              10009
lag_7              68787
lag_14            136929
lag_28            272500
roll_mean_7        10009
roll_std_7         19875
roll_max_7         10009
roll_mean_14       10009
roll_std_14        19875
roll_max_14        10009
roll_mean_28       10009
roll_std_28        19875
roll_max_28        10009
promo_lag_1        10009
promo_count_28     10009
zero_rate_28       10009
dtype: int64

Rows with ANY null: 272,500 (3.5%)


In [8]:
# Drop rows lacking full lag history

before = len(df)
df = df[df["lag_28"].notna()].copy()
after = len(df)

print(f"Dropped {before - after:,} warm-up rows ({(before-after)/before*100:.1f}%)")
print(f"Remaining: {after:,}")
print("\nNulls remaining:")
nulls = df.isna().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "None")
print("\nNew date range:", df["date"].min().date(), "to", df["date"].max().date())

Dropped 272,500 warm-up rows (3.5%)
Remaining: 7,459,611

Nulls remaining:
None

New date range: 2015-01-29 to 2017-08-15


In [9]:
df.to_parquet('../data/processed/protein_features.parquet', index=False)
print("Saved:", df.shape, "|", round(df.memory_usage(deep=True).sum()/1024**2, 1), "MB")

Saved: (7459611, 36) | 782.5 MB
